<div style="background-color: #ffffff; color: #000000; padding: 10px;">
<img src="../media/img/kisz_logo.png" width="192" height="69"> 
<h1> Working with embeddings:
<h2>An introductory workshop with applications on Semantic Search
</div>

<div style="background-color: #f6a800; color: #ffffff; padding: 10px;">
<h2>Part 3.4 - Alternative static embeddings: FastText
</div>

In this notebook, we'll explore FastText, a word embedding model developed by Facebook. FastText improves upon traditional word embeddings by representing words as bags of character n-grams, which helps it better handle out-of-vocabulary (OOV) words.

We start with some imports as usual.

In [ ]:
# imports
from gensim.models.fasttext import load_facebook_model

import nb_config
from nb_config import MODELS_PATH

from src.embeddings import get_FastText_model

import warnings
warnings.filterwarnings('ignore')

<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3>1. Overview
</div>

**FastText** is a word embedding algorithm developed by researchers at Facebook’s AI Research (FAIR) lab (paper [here](https://arxiv.org/abs/1607.04606)). Like Word2Vec, FastText is a predictive model that learns word representations by training a shallow neural network to predict context words. However, FastText introduces a significant innovation by representing each word as a bag of character-level n-grams rather than as a single atomic unit.

During training, FastText breaks each word into a collection of overlapping subwords (n-grams). For example, the word “playing” might be represented by the character n-grams <pl, pla, lay, ayi, yin, ing, ng>. The final embedding for a word is obtained by summing the embeddings of its component n-grams along with the embedding of the whole word. This subword information allows the model to learn representations that are sensitive to word morphology and shared word roots.

FastText's optimization objective is similar to Word2Vec’s skip-gram model, where the goal is to predict context words given a target word. However, the inclusion of subword information allows FastText to better capture syntactic nuances and to generate embeddings for out-of-vocabulary (OOV) words—something traditional models like Word2Vec and GloVe cannot do effectively.

We are going to use a fasttext model **wiki-news-300d-1M-subword** provided by the FastText team. This model creates embeddings with 300 dimensions and has a vocabulary of one million tokens. 

In [ ]:
# we download and prepare the model
get_FastText_model()

# we load the model
fasttext_model = load_facebook_model(MODELS_PATH+'wiki-news-300d-1M-subword.bin')

# for easy access, we assign a variable for the embeddings 
ft = fasttext_model.wv

> **NOTE**: If you have problems loading the model or it works really slow, you can load the model vectors instead. They are the same vectors that you would have gotten with the model, but we lose the ability to work with the subword patterns that are characteristic for FastText.

In [ ]:
# For loading the vectors uncomment the following lines and run the cell
# import gensim.downloader as api
# ft = api.load("fasttext-wiki-news-subwords-300")

We check that the size of the embeddings is correct and check the number of tokens contained in the vocabulary, corresponding to both words and their subwords. 

In [ ]:
embeds_shape = ft.vectors_vocab.shape

print(f"Embeddings size: {embeds_shape[1]}")
print(f"Number of (sub)words in the vocabulary: {embeds_shape[0]}")

<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3>2. Out of Vocabulary words (OOV))
</div>

We can work now with the embeddings as we did with the word2vec embeddings. The embeddings in <kbd>ft</kbd> can be accessed exactly as if they were a dictionary, using as keys the different tokens.

```
ft['<token>']
```

Let's try to get embeddings for Out of vocabulary words! This can happen if we use words that have not appeared in the training corpus, or even if known words appear misspelled. 


In [ ]:
ft['hypernanominiaturized']

In [ ]:
ft["unbelivable"]

Otherwise, **FastText** models behave much like **word2vec** models.

<div style="background-color: #b1063a; color: #ffffff; padding: 10px;">
<strong>Exercise</strong>

 Repeat the experiments we did in the notebook 3.1, now with our FastText model.

</div>

<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3>3. Advantages and disadvantages of FastText Models
</div>

Let's summarize the pros and cons of FastText embeddings, and see where can they be used.
        
#### Advantages:

> - **Handling Out-of-Vocabulary (OOV) Words**: One of FastText’s key strengths is its ability to generate embeddings for words not seen during training. Since it composes word vectors from character n-grams, it can infer representations for new words based on their subword structure, making it highly robust in dynamic language environments.
> - **Better Performance on Morphologically Rich Languages**: Languages with complex word forms (like Turkish or Finnish) benefit greatly from FastText’s subword approach. It captures meaningful representations even when words share similar roots but have different endings or prefixes.
> - **Improved Word Similarity and Classification**: FastText generally shows better performance than Word2Vec on tasks involving word similarity, text classification, and sentence representations, particularly in low-resource or noisy datasets.

#### Disadvantages:

> - **Increased Computational Complexity**: The inclusion of subword information increases the size of the training model and the computational cost compared to Word2Vec. This makes FastText somewhat slower during both training and inference, especially with large corpora.
> - **Still Static Embeddings**: Like Word2Vec, FastText generates static embeddings. Each word has one fixed vector regardless of context, which limits its ability to distinguish between different senses of the same word (polysemy).
> - **Reduced Interpretability**: Although it improves generalization, the combination of character n-grams makes it even harder to interpret what each dimension of the resulting vector represents, further decreasing transparency.

#### Applications:

> - **Text Classification**: FastText has been widely used for text classification tasks such as spam detection, topic categorization, and language identification. Its speed and accuracy make it suitable for real-time applications.
> - **Semantic Similarity & Clustering**: Similar to Word2Vec, FastText is effective in capturing semantic similarities and clustering words, but with enhanced accuracy on rare or morphologically complex words.
> - **Named Entity Recognition (NER)**: FastText embeddings contribute to better NER performance by recognizing similarities between named entities with slight variations or unseen forms, such as different spellings of a name or location.
> - **Cross-Lingual Learning**: FastText's structure facilitates the learning of cross-lingual embeddings, allowing alignment between words in different languages based on shared subword patterns, making it useful in multilingual applications and low-resource languages.